In [ ]:
!pip install langchain langchain-community faiss-cpu pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.9/515.9 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are ins

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Expt04.pdf to Expt04 (2).pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# replace with your actual file name after upload
file_name = list(uploaded.keys())[0]

loader = PyPDFLoader("Expt04 (2).pdf")
documents = loader.load()

In [ ]:
print(len(documents))
print(documents[0].page_content[:300])

1
T.Y.B.Tech CSE(AIML) Subject: Agentic Systems Lab 
  
Experiment No. : 4 
 
Title: Design and development of a Retrieval-Augmented Generation (RAG) based question 
answering chat system. 
 
Objectives: 
1. To understand RAG architecture 
2. To build a retrieval-based QA chatbot 
 
 
Key Concepts: Do


In [ ]:
!pip install -U langchain langchain-community

In [ ]:
def simple_text_splitter(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks

In [ ]:
full_text = ""

for doc in documents:
    full_text += doc.page_content + "\n"

In [ ]:
chunks = simple_text_splitter(full_text)

print("Total chunks:", len(chunks))
print(chunks[0][:300])

Total chunks: 2
T.Y.B.Tech CSE(AIML) Subject: Agentic Systems Lab 
  
Experiment No. : 4 
 
Title: Design and development of a Retrieval-Augmented Generation (RAG) based question 
answering chat system. 
 
Objectives: 
1. To understand RAG architecture 
2. To build a retrieval-based QA chatbot 
 
 
Key Concepts: Do


In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_texts(chunks, embeddings)

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 24.3 kB/s eta 0:00:00


In [ ]:
from groq import Groq

client = Groq(api_key="YOUR_API_KEY")

In [ ]:
query = "what is the main topic of the document?"

docs = vectorstore.similarity_search(query, k=3)

for i, d in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(d.page_content[:300])


--- Result 1 ---
  Stores embeddings and performs similarity search. 
5. LLM: Generates final response using retrieved context. 
 
Algorithm 
1. Install required Python libraries. 
2. Upload and load document. 
3. Load the PDF Document. 
4. Apply Text Chunking. 
5. Create embeddings. 
6. Create (FAISS) Vector Store.

--- Result 2 ---
T.Y.B.Tech CSE(AIML) Subject: Agentic Systems Lab 
  
Experiment No. : 4 
 
Title: Design and development of a Retrieval-Augmented Generation (RAG) based question 
answering chat system. 
 
Objectives: 
1. To understand RAG architecture 
2. To build a retrieval-based QA chatbot 
 
 
Key Concepts: Do


In [ ]:
def rag_answer(query):
    # Step 1: Retrieve relevant chunks
    docs = vectorstore.similarity_search(query, k=3)

    context = "\n\n".join([d.page_content for d in docs])

    # Step 2: Send to Groq LLM
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer ONLY using the given context."
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{query}
"""
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
print(rag_answer("what is this document about?"))

This document is about the design and development of a Retrieval-Augmented Generation (RAG) based question answering chat system. It outlines the objectives, key concepts, and theoretical background of the RAG architecture, as well as the components and algorithm used to implement the system.
